# 04 · Hablar como Milei · estilo con QLoRA

> Misma tesis que los labs anteriores, pero ahora la tarea es el **estilo**: coges un modelo,
> le preguntas y responde neutro (no suena a Milei), le haces QLoRA con su dataset, y vuelves a
> preguntar. Ahora habla como él (sobre todo en tono y registro).

## Qué aprendemos

1. Fine-tuning de **estilo/voz** (no una tarea con respuesta correcta, sino una forma de hablar).
2. QLoRA + Unsloth sobre Qwen3.5, **eligiendo tamaño**: 2B (T4), 9B (A100) o 27B (H100).
3. Evaluar estilo: por qué aquí el "accuracy" no sirve (proxy de vocabulario aquí, antes/después completo en el Lab 09).

## Por qué importa

Es el caso de uso "voz de marca" de la slide "Cuando SI fine-tunear": cuando necesitas que el modelo
hable SIEMPRE con un tono concreto. Y se hace exactamente igual que una tarea: datos + SFT.

## Conecta con

- **Slides de Fine-tuning · Deck 1** : "Cuando SI fine-tunear" -> tono / voz de marca.
- **Lab 03** : misma receta (QLoRA + Unsloth), pero allí tarea medible y aquí estilo.
- **Lab 05** : LLM-as-judge (gpt-5-mini puntúa respuestas) para lo que no tiene respuesta única.
- **Lab 09** : mides ESTE adapter antes/después en tres ejes (estilo, retención con ARC, toxicidad).

## Stack

Qwen via Unsloth (4-bit) · `unsloth` · `trl` · dataset `machinelearnear/multiturn_chat_milei_gpt`.
Eliges tamaño: `unsloth/Qwen3.5-2B` (mini, por defecto), `unsloth/Qwen3.5-9B` (small) o `unsloth/Qwen3.5-27B` (big).

## Hardware

- **Qwen3.5-2B** (mini) : **T4 (Colab Free)** · por defecto, para que corra sin pagar. Ejecución de referencia (oct-2026): ~33 min en total, ~22 de entrenamiento.
- **Qwen3.5-9B** : A100 (40 GB).
- **Qwen3.5-27B** : H100 (80 GB).

El dataset es pequeño (447 convs), así que el entrenamiento es corto en los tres tamaños.


---

## Setup


In [1]:
!pip install -qU unsloth datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.4/82.4 kB 4.3 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━ 21.0/28.2 MB 180.0 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.2/28.2 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━ 27.6/43.1 MB 186.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 43.1/43.1 MB 218.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/73.8 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/2.2 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 42.2 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 11.3 MB/s eta 0:00:00


In [2]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import torch
from datasets import load_dataset

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!


---

## Login en HuggingFace

Lo hacemos al principio: de aquí sacamos tu usuario para subir el adapter al final (sin escribirlo a mano).


In [3]:
import getpass
from huggingface_hub import login, whoami

login(token=getpass.getpass("HF token (write): "))
HF_USER = whoami()["name"]
print("Conectado como:", HF_USER)

Conectado como: ericrisco


---

# 1 · Elige tu Qwen · pequeño o grande

Este lab corre con tres tamaños de Qwen3.5, según la GPU que tengas en Colab. Cambia `MODEL_SIZE` y ya:
el resto del notebook es idéntico (mismo chat template y markers).

| MODEL_SIZE | Modelo | Donde corre |
|---|---|---|
| `"mini"` | Qwen3.5-2B | **T4 (Colab Free)** |
| `"small"` | Qwen3.5-9B | A100 |
| `"big"` | Qwen3.5-27B | H100 |


In [4]:
MODEL_SIZE = "mini"    # "mini" = 2B (T4 free) · "small" = 9B (A100) · "big" = 27B (H100)

MODELS = {
    "mini":  "unsloth/Qwen3.5-2B",  # ~2B · cabe en T4 free
    "small": "unsloth/Qwen3.5-9B",             # 9B · necesita A100
    "big":   "unsloth/Qwen3.5-27B",            # 27B · necesita H100
}
model_name = MODELS[MODEL_SIZE]
print("Modelo elegido:", model_name)

# Las entrevistas son LARGAS (~10k tokens de media por conversación) y se truncan a MAX_SEQ.
# En T4 usamos 1024 tokens: sobra para captar el estilo y el entrenamiento baja a ~20 min
# (Qwen3.5 mezcla atención lineal y en T4 corre en float32 sin kernels rápidos).
# Con A100/H100 puedes subir a 2048-4096 para que vea más de cada entrevista.
MAX_SEQ = 1024 if MODEL_SIZE == "mini" else 2048

model, tokenizer = FastModel.from_pretrained(
    model_name = model_name,
    max_seq_length = MAX_SEQ,
    load_in_4bit = True,
    full_finetuning = False,
)

Modelo elegido: unsloth/Qwen3.5-2B


==((====))==  Unsloth 2026.10.1: Fast Qwen3_5 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

In [7]:
tokenizer = get_chat_template(tokenizer, chat_template="qwen3")

@torch.no_grad()
def ask(question, max_new_tokens=120):
    model.eval()

    messages = [{"role": "user", "content": question}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,   # Qwen3.5 es híbrido: sin esto puede abrir un bloque <think>
    )

    inputs = tokenizer(
        text=text,                 # importante: keyword, no posicional
        return_tensors="pt",
        add_special_tokens=False,
    )

    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    return answer.strip()

---

# 2 · El "antes" · ¿suena a Milei? (no)

Le preguntamos cosas que Milei contestaria con su estilo inconfundible. El modelo base responde
correcto y neutro... pero no suena a el.


In [8]:
QUESTIONS = [
    "¿Que opinas de la casta politica?",
    "¿Que harias con el deficit fiscal y el gasto publico?",
    "Defini en una frase tu vision de la libertad.",
    "VIVA LA LIBERTAD"
]

print("=== ANTES (modelo base) ===\n")

for q in QUESTIONS:
    print("P:", q)
    print("R:", ask(q))
    print("-" * 80)

=== ANTES (modelo base) ===

P: ¿Que opinas de la casta politica?


`causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


`causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


R: La "casta política" (a menudo asociada con la **casta** en general o con figuras específicas dentro del poder) es un tema complejo y profundamente controvertido, ya que depende totalmente del contexto histórico, cultural y geográfico en el que se observe. No existe una respuesta única porque el concepto cambia según el lugar y la era.

Aquí te explico cómo se interpreta según diferentes contextos:

### 1. Contexto Histórico y Clásico (España y América Latina)
Si te refieres al sistema tradicional de "castas" como una jerarqu
--------------------------------------------------------------------------------
P: ¿Que harias con el deficit fiscal y el gasto publico?


R: La relación entre el **deficit fiscal** (o déficit presupuestario) y el **gasto público** es un tema central en la economía moderna, especialmente en el contexto de los desequilibrios económicos globales. Aquí te explico desde una perspectiva analítica y estratégica qué haría si estuviera planteado este escenario:

### 1. Análisis Causal: El Deficit como Desequilibrio de Gasto
Si observamos el **deficit fiscal** como un indicador de resultado final, no es el gasto en sí mismo lo que genera el problema, sino la
--------------------------------------------------------------------------------
P: Defini en una frase tu vision de la libertad.


R: Para mí, la libertad es la capacidad de pensamiento y acción que permite a las personas construir un mundo justo y libre de opresión, mientras mantienen la responsabilidad de cuidar el futuro de todos.
--------------------------------------------------------------------------------
P: VIVA LA LIBERTAD


R: ¡Viva la Libertad! 🎉🚀

Es un saludo enérgico y democrático. ¿Tienes alguna idea específica a la que quieras hacer referencia, o necesitas ayuda con un proyecto, una reflexión sobre el concepto o algo relacionado con la cultura de la libertad?
--------------------------------------------------------------------------------


Respuestas de manual, equilibradas, sin su vocabulario ni su tono. Vamos a darle su voz.


---

# 3 · El dataset · entrevistas de Milei

`machinelearnear/multiturn_chat_milei_gpt`: 447 conversaciones transcritas de entrevistas, ya en
formato `messages` (lista de `{role, content}`), justo lo que come un chat template.


In [10]:
ds = load_dataset("machinelearnear/multiturn_chat_milei_gpt", split="train")

print(ds)
print("Nº de conversaciones:", len(ds))
print("Turnos en la primera conversación:", len(ds[0]["messages"]))

for i, turn in enumerate(ds[0]["messages"][:6]):
    print(i, turn["role"], ":", turn["content"][:120])

README.md:   0%|          | 0.00/4.32k [00:00<?, ?B/s]

huggingface_dataset.parquet: reconstructing file:   0%|          |  0.00B / 9.91MB            

huggingface_dataset.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/447 [00:00<?, ? examples/s]

Dataset({
    features: ['channel_id', 'channel', 'uploader_url', 'video_id', 'url', 'title', 'duration', 'view_count', 'assistant', 'messages'],
    num_rows: 447
})
Nº de conversaciones: 447
Turnos en la primera conversación: 58
0 user : Mirá, en minutos va a estar también Jorge Macri.  Tenemos aquí a Javier Milei, pero vamos a preguntarle a Jorge Macri so
1 assistant :  Pero yo estoy denunciando a Gnosis.  A ver.  Allá está Jorge.  Jorge jamás haría algo así.  Jorge no haría eso.  Nosotr
2 assistant : Buenas noches y felices Pascuas.
3 user : Igualmente.  Bueno, arranquemos por... ¿Te están haciendo la campaña desde el frente de todos?
4 assistant : Bueno, la verdad es que es un pésimo gobierno.  No pueden meter ni un penal sin arquero.  Y yo creo que de a poco la gen
5 user : Javier, ¿qué pasa en la matanza?  Cuando vos ves que un gobierno que se supone defensor de los derechos humanos, al meno


In [11]:
def format_example(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

ds_text = ds.map(format_example)
print(ds_text[0]["text"][:1000])

Map:   0%|          | 0/447 [00:00<?, ? examples/s]

<|im_start|>user
Mirá, en minutos va a estar también Jorge Macri.  Tenemos aquí a Javier Milei, pero vamos a preguntarle a Jorge Macri sobre... Miren acá lo que dice Cristian Ritondo.  Juntos por el cambio debería incorporar a Javier Milei para acabar con el populismo kirchnerista.  Javier, buenas noches, bienvenido.<|im_end|>
<|im_start|>assistant
 Pero yo estoy denunciando a Gnosis.  A ver.  Allá está Jorge.  Jorge jamás haría algo así.  Jorge no haría eso.  Nosotros somos contendientes.  Que alguien me explique por qué hay una parte que me torpedea a mí.  Si el político que vos tenés enfrente es un chorro, un delincuente, un tirano, ¿qué, te vas a quedar quietito?  ¿Vas a dejar que te violen?<|im_end|>
<|im_start|>assistant
Buenas noches y felices Pascuas.<|im_end|>
<|im_start|>user
Igualmente.  Bueno, arranquemos por... ¿Te están haciendo la campaña desde el frente de todos?<|im_end|>
<|im_start|>assistant
Bueno, la verdad es que es un pésimo gobierno.  No pueden meter ni un penal 

---

# 4 · QLoRA + entrenamiento


In [12]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

In [13]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds_text,
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = MAX_SEQ,        # trunca cada entrevista (ver celda del modelo)
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,        # dataset pequeño (447) -> dos pasadas
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)

# Entrenar solo sobre lo que dice el assistant (la voz de Milei), no sobre el turno del usuario
# Markers ChatML de Qwen
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)
trainer.train()

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/447 [00:00<?, ? examples/s]

Map:   0%|          | 0/447 [00:00<?, ? examples/s]

Filter:   0%|          | 0/447 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 441 | Num Epochs = 2 | Total steps = 112
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 16,819,200 of 2,230,060,864 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,3.168571
20,2.995486
30,3.001101
40,3.109719
50,3.064784
60,3.074732
70,2.952549
80,2.974858
90,2.899953
100,2.948199


TrainOutput(global_step=112, training_loss=3.0140243896416257, metrics={'train_runtime': 1483.432, 'train_samples_per_second': 0.595, 'train_steps_per_second': 0.076, 'total_flos': 9316045378289664.0, 'train_loss': 3.0140243896416257, 'epoch': 2.0})

---

# 5 · El "después" · ahora habla como Milei

Las MISMAS preguntas del paso 2.

Qué esperar con el `mini` en T4 (2B, 1024 tokens por entrevista, 2 epochs): lo que más cambia es el
**registro**. Las respuestas pasan de "asistente de manual" (largas, con negritas y listas) a frases cortas,
orales, en primera persona y a la contra, como en una entrevista. Las ideas y muletillas concretas se
marcan más con un modelo mayor (`small`/`big`) y más contexto (`MAX_SEQ`). Hay sampling: cada ejecución varía.


In [14]:
print("=== DESPUES (fine-tuneado) ===\n")
for q in QUESTIONS:
    print("P:", q)
    print("R:", ask(q), "\n")

=== DESPUES (fine-tuneado) ===

P: ¿Que opinas de la casta politica?


R: No, no, no, no, no. 

P: ¿Que harias con el deficit fiscal y el gasto publico?


R: Y yo no voy a hablar de eso porque es una cuestión que no es mía. 

P: Defini en una frase tu vision de la libertad.


R: No me parece que la libertad es un concepto que existe.  La libertad es un problema que la gente tiene que resolver, que es una cuestión de valor.  Por lo tanto, te lo voy a dar a tu gusto. 

P: VIVA LA LIBERTAD


R: El día que la sociedad se reconecte con el derecho a la libertad, la libertad es el derecho más grande que tenemos como humanos. 



---

# 6 · ¿Cómo se evalua el estilo?

El estilo no tiene "respuesta correcta", así que la accuracy no sirve. Dos enfoques:

1. **Proxy rápido** : contar marcadores de su vocabulario (no es ciencia, pero da una señal).
2. **LLM-as-judge** (lo serio) : otro modelo puntúa "¿suena a Milei?" de 1 a 5. Es la técnica que usas en el Lab 05 (gpt-5-mini como juez); en el Lab 09 mides este adapter antes/después en varios ejes.

Aquí hacemos el proxy rápido, **antes y después** sobre el mismo modelo: `disable_adapter()` apaga el LoRA y vuelves al Qwen base sin recargar nada.

Lo que salió en la ejecución de referencia (T4, `mini`): formato chatbot **50% -> 0%** y respuestas de
~66 a ~37 palabras (registro oral, como en la entrevista), pero los marcadores de vocabulario **bajaron**
(3,8 -> 1,4 por 100 palabras). No es que el modelo base "suene más a Milei": es que repite las palabras de
la pregunta ("casta", "déficit", "libertad") dentro de respuestas largas. **Un proxy mal diseñado engaña.**
En el Lab 09 lo corregimos (no contamos los marcadores que ya vienen en la pregunta) y medimos además
retención y toxicidad.


In [15]:
MARKERS = ["casta", "libertad", "zurdo", "carajo", "kuka", "deficit",
           "estado", "liberal", "socialis", "viva la"]

import re

def style_stats(answers):
    """Dos señales baratas sobre un lote de respuestas:
    - marcadores de su vocabulario por cada 100 palabras (densidad, no total: si no, gana
      la respuesta más larga, y el modelo base escribe MUCHO más);
    - % de respuestas con formato de chatbot (negritas, títulos, listas), que Milei no usa al hablar."""
    words = sum(len(a.split()) for a in answers)
    hits = sum(a.lower().count(m) for a in answers for m in MARKERS)
    chatbot = sum(bool(re.search(r"\*\*|^#+ |^\s*[-*] |^\s*\d+\. ", a, re.M)) for a in answers)
    return 100 * hits / max(words, 1), 100 * chatbot / len(answers), words / len(answers)

with model.disable_adapter():   # adapter apagado = el Qwen base de antes
    base_answers = [ask(q) for q in QUESTIONS]
ft_answers = [ask(q) for q in QUESTIONS]

for name, answers in [("base", base_answers), ("fine-tuneado", ft_answers)]:
    dens, fmt, n = style_stats(answers)
    print(f"{name:13s} marcadores/100 palabras: {dens:4.1f} · formato chatbot: {fmt:3.0f}% · palabras/resp: {n:5.1f}")
print("\n(Proxy tosco y con sampling: varía entre ejecuciones. Para evaluar estilo en serio -> LLM-as-judge)")

base          marcadores/100 palabras:  3.8 · formato chatbot:  50% · palabras/resp:  66.5
fine-tuneado  marcadores/100 palabras:  1.4 · formato chatbot:   0% · palabras/resp:  37.0

(Proxy tosco y con sampling: varía entre ejecuciones. Para evaluar estilo en serio -> LLM-as-judge)


---

# Cierre · lo que nos llevamos

- Fine-tuning de **estilo**: misma receta que una tarea (datos + QLoRA), pero el objetivo es la VOZ.
- Pasamos a un Qwen3.5 moderno (2B por defecto, hasta 27B cambiando `MODEL_SIZE`): con QLoRA el 2B entrena en una T4 gratis.
- Y vimos que **evaluar estilo es otro juego**: no hay accuracy; el proxy de vocabulario da una señal y lo serio es un LLM-as-judge.

El adapter de Milei es, como en el Lab 2, una capa que pones y quitas: podrias tener varios estilos
sobre el mismo modelo base.

## Hacia el alignment

Hasta aquí el modelo aprende TAREAS y ESTILOS. Pero, ¿se comporta como queremos (util, honesto,
inofensivo)? Eso es alignment: el siguiente bloque (Reward Model + RLHF, luego DPO/ORPO).


---

## Guarda tu adapter (para el Lab 09)

Sube el adapter de Milei para poder evaluarlo (antes/después) en el capstone de evaluación.


In [16]:
ADAPTER_REPO = f"{HF_USER}/milei-qlora-adapter"   # usa tu usuario del login del inicio
model.push_to_hub(ADAPTER_REPO)       # sube solo el adapter (pesa poco)
tokenizer.push_to_hub(ADAPTER_REPO)
print("adapter guardado en", ADAPTER_REPO)

Unsloth: Restored added_tokens_decoder metadata in ericrisco/milei-qlora-adapter/tokenizer_config.json.


adapter guardado en ericrisco/milei-qlora-adapter
